# Monitoring

## Thomas: Shared setup and input provenance

In [12]:
import os
os.environ.setdefault("SAGEMAKER_SUPPRESS_V2_WARNING", "1")
import io, json, time
from datetime import datetime, timezone
from urllib.parse import urlparse
import boto3
from IPython.display import display, Markdown

session = boto3.Session()
region = session.region_name or "us-east-1"
account_id = session.client("sts", region_name=region).get_caller_identity()["Account"]
s3 = session.client("s3", region_name=region)
sm = session.client("sagemaker", region_name=region)
cw = session.client("cloudwatch", region_name=region)

role_arn = os.environ.get("SAGEMAKER_ROLE_ARN")
if not role_arn:
    try:
        from sagemaker.core.helper.session_helper import get_execution_role
    except ImportError:
        from sagemaker import get_execution_role
    role_arn = get_execution_role()

bucket = f"sagemaker-aai540-modeling-{account_id}"
root = f"flight-delay-xgboost"
TRAIN_S3_URI = f"s3://{bucket}/{root}/data-uncompressed/train/data.csv"
CURRENT_FEATURES_S3_URI = f"s3://{bucket}/{root}/data-uncompressed/test/data.csv"
PREDICTIONS_S3_URI = None  # auto-discover latest original completed Batch Transform; or set exact .out S3 URI
TRANSFORM_JOB_NAME = None   # set original transform job name to pin a specific run
TRAINING_JOB_NAME = None    # optional original training job name
SKLEARN_IMAGE_URI = None    # set only if image_uris.retrieve is unavailable in your lab
PROJECT = "FlightDelayPrediction"
NAMESPACE = "FlightDelay/Monitoring"

FEATURES = ["month", "day_of_month", "day_of_week", "is_weekend",
            "scheduled_departure_hour", "scheduled_arrival_hour",
            "scheduled_duration", "distance"]

def parse_s3(uri):
    parsed = urlparse(uri)
    if parsed.scheme != "s3" or not parsed.netloc or not parsed.path.lstrip("/"):
        raise ValueError(f"Expected nonempty s3://bucket/key URI: {uri}")
    return parsed.netloc, parsed.path.lstrip("/")

def objects(uri):
    b, key = parse_s3(uri)
    paginator = s3.get_paginator("list_objects_v2")
    return [f"s3://{b}/{item['Key']}" for page in paginator.paginate(Bucket=b, Prefix=key)
            for item in page.get("Contents", [])]

def require_object(uri):
    b, key = parse_s3(uri)
    s3.head_object(Bucket=b, Key=key)
    return uri

def latest_job(kind, prefix, status_key):
    method = sm.get_paginator(f"list_{kind}_jobs")
    items = [item for page in method.paginate(NameContains=prefix, SortBy="CreationTime",
              SortOrder="Descending") for item in page.get(f"{kind.title()}JobSummaries", [])
              if item[status_key] == "Completed" and item[f"{kind.title()}JobName"].startswith(prefix)]
    if not items:
        raise RuntimeError(f"No completed {kind} job starts with {prefix}. Set an override above.")
    return items[0][f"{kind.title()}JobName"]

if TRANSFORM_JOB_NAME is None:
    TRANSFORM_JOB_NAME = latest_job("transform", "flight-delay-test-", "TransformJobStatus")
transform = sm.describe_transform_job(TransformJobName=TRANSFORM_JOB_NAME)
if transform["TransformJobStatus"] != "Completed":
    raise RuntimeError(f"Selected transform is {transform['TransformJobStatus']}, not Completed")
if PREDICTIONS_S3_URI is None:
    candidates = [uri for uri in objects(transform["TransformOutput"]["S3OutputPath"])
                  if uri.endswith(".out")]
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one .out file; found {len(candidates)}. Set PREDICTIONS_S3_URI explicitly.")
    PREDICTIONS_S3_URI = candidates[0]
require_object(TRAIN_S3_URI)
require_object(CURRENT_FEATURES_S3_URI)
require_object(PREDICTIONS_S3_URI)
if TRAINING_JOB_NAME is None:
    original_model = transform["ModelName"]
    if not original_model.startswith("flight-delay-xgboost-"):
        raise RuntimeError(f"Selected transform uses {original_model}, not the original XGBoost model")
    TRAINING_JOB_NAME = original_model

if SKLEARN_IMAGE_URI is None:
    try:
        from sagemaker.core import image_uris
    except ImportError:
        from sagemaker import image_uris
    SKLEARN_IMAGE_URI = image_uris.retrieve("sklearn", region, "1.2-1",
                                            instance_type="ml.m5.large")

print("Region:", region, "Role:", role_arn)
print("Training baseline:", TRAIN_S3_URI)
print("Current features:", CURRENT_FEATURES_S3_URI)
print("Labeled predictions:", PREDICTIONS_S3_URI)
print("Transform:", TRANSFORM_JOB_NAME, "Training:", TRAINING_JOB_NAME)
print("Processing image:", SKLEARN_IMAGE_URI)

[09/28/26 22:00:10] INFO     Defaulting to only available Python version: py3                     ]8;id=3769559;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=3769560;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\

Region: us-east-1 Role: arn:aws:iam::944832759247:role/LabRole
Training baseline: s3://sagemaker-aai540-modeling-944832759247/flight-delay-xgboost/data-uncompressed/train/data.csv
Current features: s3://sagemaker-aai540-modeling-944832759247/flight-delay-xgboost/data-uncompressed/test/data.csv
Labeled predictions: s3://sagemaker-aai540-modeling-944832759247/flight-delay-xgboost/predictions/test/flight-delay-test-20260925-162729/data.csv.out
Transform: flight-delay-test-20260925-162729 Training: flight-delay-xgboost-20260925-162308
Processing image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3


### Shared managed-job helper

In [13]:
from uuid import uuid4

def processing_input(name, uri, local_path):
    return {"InputName": name, "S3Input": {
        "S3Uri": uri, "LocalPath": local_path, "S3DataType": "S3Prefix",
        "S3InputMode": "File", "S3DataDistributionType": "FullyReplicated"}}

def submit_monitor(kind, script, inputs):
    run_stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:4]
    name = f"flight-w5-{kind}-{run_stamp}"
    code_key = f"{root}/week5/code/{kind}-{run_stamp}.py"
    s3.put_object(Bucket=bucket, Key=code_key, Body=script.encode("utf-8"),
                  ContentType="text/x-python", ServerSideEncryption="AES256")
    code_uri = f"s3://{bucket}/{code_key}"
    output_uri = f"s3://{bucket}/{root}/week5/reports/{kind}/{name}/"
    all_inputs = [processing_input("code", code_uri, "/opt/ml/processing/code")]
    all_inputs += [processing_input(label, uri, f"/opt/ml/processing/input/{label}")
                   for label, uri in inputs.items()]
    sm.create_processing_job(
        ProcessingJobName=name, RoleArn=role_arn,
        AppSpecification={"ImageUri": SKLEARN_IMAGE_URI,
                          "ContainerEntrypoint": ["python3", f"/opt/ml/processing/code/{kind}-{run_stamp}.py"]},
        ProcessingInputs=all_inputs,
        ProcessingOutputConfig={"Outputs": [{"OutputName": "reports", "S3Output": {
            "S3Uri": output_uri, "LocalPath": "/opt/ml/processing/output",
            "S3UploadMode": "EndOfJob"}}]},
        ProcessingResources={"ClusterConfig": {"InstanceCount": 1,
            "InstanceType": "ml.m5.large", "VolumeSizeInGB": 20}},
        StoppingCondition={"MaxRuntimeInSeconds": 3600},
        Tags=[{"Key": "Project", "Value": PROJECT},
              {"Key": "Owner", "Value": "Denis" if kind == "model" else "Daniel"}])
    print("Submitted", name, "→", output_uri)
    while True:
        state = sm.describe_processing_job(ProcessingJobName=name)
        if state["ProcessingJobStatus"] in ("Completed", "Failed", "Stopped"):
            break
        time.sleep(30)
    if state["ProcessingJobStatus"] != "Completed":
        raise RuntimeError(f"{name}: {state['ProcessingJobStatus']}: {state.get('FailureReason', 'See CloudWatch logs')}")
    return name, output_uri

def read_json(uri):
    b, key = parse_s3(uri)
    return json.loads(s3.get_object(Bucket=b, Key=key)["Body"].read())

def publish_metric(metric, value, unit="None"):
    cw.put_metric_data(Namespace=NAMESPACE, MetricData=[{
        "MetricName": metric, "Dimensions": [{"Name": "Project", "Value": PROJECT}],
        "Timestamp": datetime.now(timezone.utc), "Value": float(value), "Unit": unit}])

def create_alarm(name, metric, comparison, threshold):
    cw.put_metric_alarm(
        AlarmName=f"FlightDelay-{name}",
        AlarmDescription=f"Week 5 {PROJECT} batch check: {metric}",
        Namespace=NAMESPACE, MetricName=metric,
        Dimensions=[{"Name": "Project", "Value": PROJECT}],
        Statistic="Average", Period=300, EvaluationPeriods=1,
        ComparisonOperator=comparison, Threshold=threshold,
        TreatMissingData="notBreaching", ActionsEnabled=False
    )

## Denis: Model quality monitoring and model report

In [14]:
MODEL_SCRIPT = r'''
import json
from pathlib import Path
from html import escape
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix)
input_dir = Path("/opt/ml/processing/input/predictions")
files = sorted(input_dir.rglob("*.out"))
if len(files) != 1:
    raise RuntimeError(f"Expected exactly one labeled .out file; found {files}")
df = pd.read_csv(files[0], header=None)
if len(df) == 0 or df.shape[1] != 2:
    raise ValueError("Predictions must contain actual label and probability")
df.columns = ["actual", "probability"]
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="raise")
if df.isna().any().any() or not df.actual.isin([0, 1]).all() or not df.probability.between(0, 1).all():
    raise ValueError("Invalid labels, probabilities, or missing values")
y = df.actual.astype(int)
p = df.probability
pred = (p >= 0.5).astype(int)
cm = confusion_matrix(y, pred, labels=[0, 1])
metrics = {"accuracy": accuracy_score(y, pred),
           "balanced_accuracy": balanced_accuracy_score(y, pred),
           "delayed_precision": precision_score(y, pred, zero_division=0),
           "delayed_recall": recall_score(y, pred, zero_division=0),
           "delayed_f1": f1_score(y, pred, zero_division=0),
           "roc_auc": roc_auc_score(y, p) if y.nunique() == 2 else None,
           "pr_auc": average_precision_score(y, p) if y.nunique() == 2 else None}
metrics = {k: (None if v is None else float(v)) for k, v in metrics.items()}
reference = {"delayed_recall": 0.7330, "pr_auc": 0.4000}
minimum = {"delayed_recall": 0.6830, "pr_auc": 0.3700}
alerts = [f"{key} below {value:.4f}" for key, value in minimum.items()
          if metrics[key] is None or metrics[key] < value]
report = {"records": int(len(df)), "delayed_count": int(y.sum()),
          "delayed_prevalence": float(y.mean()), "threshold": 0.5,
          "metrics": metrics, "reference_week4": reference, "minimum_acceptable": minimum,
          "confusion_matrix": {"tn": int(cm[0, 0]), "fp": int(cm[0, 1]),
                               "fn": int(cm[1, 0]), "tp": int(cm[1, 1])},
          "alerts": alerts, "status": "CHECK" if alerts else "PASS"}
out = Path("/opt/ml/processing/output")
out.mkdir(parents=True, exist_ok=True)
(out / "model_report.json").write_text(json.dumps(report, indent=2))
rows = "".join(f"<tr><th>{escape(k)}</th><td>{v if v is not None else 'undefined (one label class)'}</td></tr>"
               for k, v in metrics.items())
(out / "model_report.html").write_text("<h1>Flight delay: model quality</h1><p>" +
    escape(report["status"] + " | " + ", ".join(alerts)) + "</p><table>" + rows +
    "</table><pre>" + escape(json.dumps(report["confusion_matrix"], indent=2)) + "</pre>")
print(json.dumps(report, indent=2))
'''
compile(MODEL_SCRIPT, '<MODEL_SCRIPT>', 'exec')
print('Monitoring script ready: '+str(len(MODEL_SCRIPT.splitlines()))+' lines')

Monitoring script ready: 52 lines


### Run monitor

In [15]:
model_job, model_output = submit_monitor(
    "model", MODEL_SCRIPT, {"predictions": PREDICTIONS_S3_URI}
)
model_report = read_json(model_output + "model_report.json")
print("Model Processing job:", model_job)
print("SageMaker model report:", model_output + "model_report.html")

Submitted flight-w5-model-20260928-220010-f238 → s3://sagemaker-aai540-modeling-944832759247/flight-delay-xgboost/week5/reports/model/flight-w5-model-20260928-220010-f238/
Model Processing job: flight-w5-model-20260928-220010-f238
SageMaker model report: s3://sagemaker-aai540-modeling-944832759247/flight-delay-xgboost/week5/reports/model/flight-w5-model-20260928-220010-f238/model_report.html


### Review results

In [16]:
import pandas as pd

model_summary = pd.DataFrame([
    {"measure": key, "observed": value,
     "week4_reference": model_report["reference_week4"].get(key),
     "minimum_acceptable": model_report["minimum_acceptable"].get(key)}
    for key, value in model_report["metrics"].items()
])
display(model_summary.round(4))
display(pd.DataFrame([model_report["confusion_matrix"]]))
print("Labeled records:", f"{model_report['records']:,}")
print("Model quality:", model_report["status"])
for alert in model_report["alerts"]:
    print("Investigate:", alert)

,measure,observed,week4_reference,minimum_acceptable
0,accuracy,0.6032,NaN,NaN
1,balanced_accuracy,0.6418,NaN,NaN
2,delayed_precision,0.3985,NaN,NaN
3,delayed_recall,0.7330,0.733,0.683
4,delayed_f1,0.5163,NaN,NaN
5,roc_auc,0.6709,NaN,NaN
6,pr_auc,0.4000,0.400,0.370


,tn,fp,fn,tp
0,239908,195891,47257,129755


Labeled records: 612,811
Model quality: PASS


### CloudWatch model alert

In [17]:
publish_metric("ModelDelayedRecall", model_report["metrics"]["delayed_recall"])
if model_report["metrics"]["pr_auc"] is not None:
    publish_metric("ModelPRAUC", model_report["metrics"]["pr_auc"])
create_alarm("DelayedRecall", "ModelDelayedRecall", "LessThanThreshold", 0.683)
print("Model metrics published; delayed-recall alarm configured.")

Model metrics published; delayed-recall alarm configured.


## Daniel: Data quality monitoring and data report

In [18]:
DATA_SCRIPT = r'''

import json
from pathlib import Path
from html import escape
import numpy as np
import pandas as pd
features = ["month", "day_of_month", "day_of_week", "is_weekend",
            "scheduled_departure_hour", "scheduled_arrival_hour",
            "scheduled_duration", "distance"]
columns = ["delayed"] + features
bounds = {"month": (1, 12), "day_of_month": (1, 31), "day_of_week": (0, 7),
          "is_weekend": (0, 1), "scheduled_departure_hour": (0, 24),
          "scheduled_arrival_hour": (0, 24), "scheduled_duration": (0, None),
          "distance": (0, None)}
def load(label):
    files = sorted(Path(f"/opt/ml/processing/input/{label}").rglob("*.csv"))
    if len(files) != 1:
        raise RuntimeError(f"Expected one CSV for {label}; found {files}")
    df = pd.read_csv(files[0], header=None)
    if df.shape[1] != len(columns) or df.empty:
        raise ValueError(f"{label} must contain 9 nonempty, headerless columns; got {df.shape}")
    df.columns = columns
    return df[features].apply(pd.to_numeric, errors="coerce"), df[features].isna()
def psi(ref, cur):
    a = ref.dropna().to_numpy(dtype=float)
    b = cur.dropna().to_numpy(dtype=float)
    if not len(a) or not len(b):
        return None
    values = np.unique(a)
    if len(values) <= 12 and np.allclose(values, np.round(values)):
        # Calendar and binary inputs are discrete. Count every value seen in
        # either batch, including categories absent from the reference.
        categories = np.union1d(values, np.unique(b))
        ref_counts = np.array([(a == v).sum() for v in categories])
        cur_counts = np.array([(b == v).sum() for v in categories])
    else:
        quantiles = np.unique(np.quantile(a, np.linspace(0, 1, 11)))
        if len(quantiles) == 1:
            categories = np.union1d(values, np.unique(b))
            ref_counts = np.array([(a == v).sum() for v in categories])
            cur_counts = np.array([(b == v).sum() for v in categories])
        else:
            midpoints = quantiles[:-1] + (quantiles[1:] - quantiles[:-1]) / 2
            edges = np.r_[-np.inf, midpoints, np.inf]
            ref_counts = np.histogram(a, bins=edges)[0]
            cur_counts = np.histogram(b, bins=edges)[0]
    # Jeffreys smoothing keeps PSI finite for new or vanishing categories.
    k = len(ref_counts)
    x = (ref_counts + 0.5) / (len(a) + 0.5 * k)
    y = (cur_counts + 0.5) / (len(b) + 0.5 * k)
    return float(np.sum((y - x) * np.log(y / x)))
ref, _ = load("reference")
cur, _ = load("current")
checks = {}
for name in features:
    low, high = bounds[name]
    values = cur[name]
    invalid = values.notna() & ((values < low) | ((values > high) if high is not None else False))
    checks[name] = {"reference_mean": None if ref[name].notna().sum() == 0 else float(ref[name].mean()),
                    "current_mean": None if values.notna().sum() == 0 else float(values.mean()),
                    "missing_fraction": float(values.isna().mean()),
                    "invalid_fraction": float(invalid.mean()),
                    "psi": psi(ref[name], values)}
alerts = []
for name, row in checks.items():
    if row["psi"] is None or row["psi"] >= 0.20:
        alerts.append(f"{name}: drift PSI >= 0.20 or unavailable")
    if row["missing_fraction"] > 0.01 or row["invalid_fraction"] > 0.01:
        alerts.append(f"{name}: missing or invalid > 1%")
report = {"reference_rows": int(len(ref)), "current_rows": int(len(cur)),
          "schema": columns, "psi_alert_threshold": 0.20,
          "feature_checks": checks, "drifted_features": sum(
              row["psi"] is None or row["psi"] >= 0.20 for row in checks.values()),
          "max_missing_fraction": max(row["missing_fraction"] for row in checks.values()),
          "alerts": alerts, "status": "CHECK" if alerts else "PASS"}
out = Path("/opt/ml/processing/output")
out.mkdir(parents=True, exist_ok=True)
(out / "data_report.json").write_text(json.dumps(report, indent=2))
rows = "".join("<tr><th>" + escape(name) + "</th>" + "".join(
    f"<td>{values[key]}</td>" for key in ("reference_mean", "current_mean",
                                         "missing_fraction", "invalid_fraction", "psi")) + "</tr>"
    for name, values in checks.items())
(out / "data_report.html").write_text("<h1>Flight delay: data quality</h1><p>" +
    escape(report["status"] + " | " + ", ".join(alerts)) +
    "</p><table><tr><th>Feature</th><th>Reference mean</th><th>Current mean</th>" +
    "<th>Missing</th><th>Invalid</th><th>PSI</th></tr>" + rows + "</table>")
print(json.dumps(report, indent=2))
'''
compile(DATA_SCRIPT, '<DATA_SCRIPT>', 'exec')
print('Monitoring script ready:', len(DATA_SCRIPT.splitlines()), 'lines')

Monitoring script ready: 88 lines


### Run monitor

In [19]:
data_job, data_output = submit_monitor(
    "data", DATA_SCRIPT,
    {"reference": TRAIN_S3_URI, "current": CURRENT_FEATURES_S3_URI}
)
data_report = read_json(data_output + "data_report.json")
print("Data Processing job:", data_job)
print("SageMaker data report:", data_output + "data_report.html")


Submitted flight-w5-data-20260928-220541-2f21 → s3://sagemaker-aai540-modeling-944832759247/flight-delay-xgboost/week5/reports/data/flight-w5-data-20260928-220541-2f21/
Data Processing job: flight-w5-data-20260928-220541-2f21
SageMaker data report: s3://sagemaker-aai540-modeling-944832759247/flight-delay-xgboost/week5/reports/data/flight-w5-data-20260928-220541-2f21/data_report.html


### Review results

In [20]:
data_summary = pd.DataFrame.from_dict(
    data_report["feature_checks"], orient="index"
).rename_axis("feature").reset_index()
data_summary["drift_flag"] = data_summary["psi"].isna() | (data_summary["psi"] >= 0.20)
data_summary["quality_flag"] = (
    (data_summary["missing_fraction"] > 0.01)
    | (data_summary["invalid_fraction"] > 0.01)
)
display(data_summary.round(4))
print("Current batch rows:", f"{data_report['current_rows']:,}")
print("Drifted features:", data_report["drifted_features"])
print("Data quality:", data_report["status"])
for alert in data_report["alerts"]:
    print("Investigate:", alert)

,feature,reference_mean,current_mean,missing_fraction,invalid_fraction,psi,drift_flag,quality_flag
0,month,3.0832,7.0000,0.0,0.0,27.9455,True,False
1,day_of_month,15.7108,16.1683,0.0,0.0,0.0076,False,False
2,day_of_week,4.0149,3.8994,0.0,0.0,0.0146,False,False
3,is_weekend,0.2772,0.2544,0.0,0.0,0.0027,False,False
4,scheduled_departure_hour,12.9632,12.9923,0.0,0.0,0.0024,False,False
5,scheduled_arrival_hour,14.6615,14.4701,0.0,0.0,0.0031,False,False
6,scheduled_duration,150.0323,148.9239,0.0,0.0,0.0037,False,False
7,distance,847.3628,858.0494,0.0,0.0,0.0038,False,False


Current batch rows: 612,811
Drifted features: 1
Data quality: CHECK
Investigate: month: drift PSI >= 0.20 or unavailable


### CloudWatch data alert

In [21]:
publish_metric("DataDriftedFeatures", data_report["drifted_features"], "Count")
publish_metric("DataMissingRatePct", data_report["max_missing_fraction"] * 100, "Percent")
create_alarm("DataDrift", "DataDriftedFeatures", "GreaterThanOrEqualToThreshold", 1)
print("Data metrics published; drift alarm configured.")

Data metrics published; drift alarm configured.


## Combined team effort: Infrastructure monitor and CloudWatch dashboard

Team contributions
- Thomas Geraci: Set up the shared SageMaker inputs and execution role, checked the training, Batch Transform, and Processing job statuses, published job health, and assembled the CloudWatch dashboard.
- Denis Mulabegovic: Implemented the model-quality Processing check, generated the labeled-batch model report, reviewed recall, PR AUC, and confusion counts, and published model metrics with a recall alarm.
- Daniel Sims: Implemented the data-quality Processing check, generated the reference-versus-current data report, reviewed feature PSI and missing or invalid values, and published data metrics with a drift alarm.
The dashboard combines all three monitoring streams. The month drift flag is documented for team review; it does not change the completed job statuses.

In [22]:
jobs = [
    ("training", TRAINING_JOB_NAME, "TrainingJobStatus"),
    ("transform", TRANSFORM_JOB_NAME, "TransformJobStatus"),
    ("processing", model_job, "ProcessingJobStatus"),
    ("processing", data_job, "ProcessingJobStatus"),
]
if model_report["records"] != data_report["current_rows"]:
    raise ValueError("Prediction row count differs from current feature rows; choose matching batches")
job_statuses = {}
for kind, name, status_field in jobs:
    info = getattr(sm, f"describe_{kind}_job")(**{f"{kind.title()}JobName": name})
    job_statuses[name] = info[status_field]
    print(kind, name, info[status_field], info.get("FailureReason", ""))
healthy = all(state == "Completed" for state in job_statuses.values())
publish_metric("JobHealth", int(healthy))
create_alarm("JobHealth", "JobHealth", "LessThanThreshold", 1)

metric = lambda name: [NAMESPACE, name, "Project", PROJECT]
widgets = [
    {"type": "text", "x": 0, "y": 0, "width": 24, "height": 2,
     "properties": {"markdown": "# Flight delay batch monitoring\nCompleted job snapshots; rerun for new labeled batches."}},
    {"type": "metric", "x": 0, "y": 2, "width": 12, "height": 6,
     "properties": {"region": region, "title": "Model quality", "metrics": [metric("ModelDelayedRecall"), metric("ModelPRAUC")], "stat": "Average", "period": 300}},
    {"type": "metric", "x": 12, "y": 2, "width": 12, "height": 6,
     "properties": {"region": region, "title": "Data quality", "metrics": [metric("DataDriftedFeatures"), metric("DataMissingRatePct")], "stat": "Average", "period": 300}},
    {"type": "metric", "x": 0, "y": 8, "width": 12, "height": 6,
     "properties": {"region": region, "title": "Batch job health", "metrics": [metric("JobHealth")], "stat": "Minimum", "period": 300, "yAxis": {"left": {"min": 0, "max": 1}}}},
    {"type": "metric", "x": 12, "y": 8, "width": 12, "height": 6,
     "properties": {"region": region, "title": "SageMaker CPU (recent jobs)",
                    "metrics": [ [{"expression": "SEARCH('{/aws/sagemaker/ProcessingJobs,Host} MetricName=\"CPUUtilization\"', 'Average', 300)", "id": "cpup"}],
 [{"expression": "SEARCH('{/aws/sagemaker/TrainingJobs,Host} MetricName=\"CPUUtilization\"', 'Average', 300)", "id": "cput"}],
 [{"expression": "SEARCH('{/aws/sagemaker/TransformJobs,Host} MetricName=\"CPUUtilization\"', 'Average', 300)", "id": "cpub"}]],
                    "period": 300}},
    {"type": "metric", "x": 0, "y": 14, "width": 12, "height": 6,
     "properties": {"region": region, "title": "SageMaker memory (recent jobs)",
                    "metrics": [ [{"expression": "SEARCH('{/aws/sagemaker/ProcessingJobs,Host} MetricName=\"MemoryUtilization\"', 'Average', 300)", "id": "memp"}],
 [{"expression": "SEARCH('{/aws/sagemaker/TrainingJobs,Host} MetricName=\"MemoryUtilization\"', 'Average', 300)", "id": "memt"}],
 [{"expression": "SEARCH('{/aws/sagemaker/TransformJobs,Host} MetricName=\"MemoryUtilization\"', 'Average', 300)", "id": "memb"}]],
                    "period": 300}},
]
dashboard_name = "FlightDelay-Week5-Batch-Monitoring"
response = cw.put_dashboard(DashboardName=dashboard_name,
                            DashboardBody=json.dumps({"widgets": widgets}))
if response.get("DashboardValidationMessages"):
    raise RuntimeError(response["DashboardValidationMessages"])
link = f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={dashboard_name}"
print("Dashboard:", link)
print("Model report:", model_output + "model_report.html")
print("Data report:", data_output + "data_report.html")
print("Statuses:", job_statuses)

training flight-delay-xgboost-20260925-162308 Completed 
transform flight-delay-test-20260925-162729 Completed 
processing flight-w5-model-20260928-220010-f238 Completed 
processing flight-w5-data-20260928-220541-2f21 Completed 
Dashboard: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=FlightDelay-Week5-Batch-Monitoring
Model report: s3://sagemaker-aai540-modeling-944832759247/flight-delay-xgboost/week5/reports/model/flight-w5-model-20260928-220010-f238/model_report.html
Data report: s3://sagemaker-aai540-modeling-944832759247/flight-delay-xgboost/week5/reports/data/flight-w5-data-20260928-220541-2f21/data_report.html
Statuses: {'flight-delay-xgboost-20260925-162308': 'Completed', 'flight-delay-test-20260925-162729': 'Completed', 'flight-w5-model-20260928-220010-f238': 'Completed', 'flight-w5-data-20260928-220541-2f21': 'Completed'}
